In [22]:
import numpy as np
import pandas as pd
import plotly.express as px

In [23]:
FWD_MS_COL = "forward_ms"
BWD_MS_COL = "backward_ms"
FWD_MEM_COL = "forward_memory_mb"
BWD_MEM_COL = "backward_memory_mb"
MS_TOTAL = "ms_total"

CONV_COL = "conv_type"
DATASET_COL = "dataset"
BACKEND_COL = "backend"
HEADS_COL = "heads"
FEATURE_DIM_COL = "feature_dim"

In [24]:
REORDERING_COL = "graph_reordering_partition_size"

# other columns used for tuning parameters

In [25]:
BASELINE = "dgl"

def add_speedup_columns(df_in: pd.DataFrame, baseline: str = BASELINE) -> pd.DataFrame:
    """
    For each (dataset, feature_dim, heads) triple, compute speedup
    vs `baseline` separately for forward_ms and backward_ms.
    """
    df = df_in.copy()
    idx_cols = [CONV_COL, DATASET_COL, FEATURE_DIM_COL, HEADS_COL, REORDERING_COL]

    for metric in [FWD_MS_COL, BWD_MS_COL, FWD_MEM_COL, BWD_MEM_COL, MS_TOTAL]:
        base_col = f"{metric}_baseline_{baseline}"
        speed_col = f"{metric}_speedup_vs_{baseline}"
        df[base_col] = np.nan
        df[speed_col] = np.nan

        for key, group in df.groupby(idx_cols):
            base_rows = group[group["backend"] == baseline]
            if base_rows.empty:
                continue
            base_val = base_rows.iloc[0][metric]
            if not np.isfinite(base_val) or base_val <= 0:
                continue

            mask = (
                (df[CONV_COL] == key[0]) &
                (df[DATASET_COL] == key[1]) &
                (df[FEATURE_DIM_COL] == key[2]) &
                (df[HEADS_COL] == key[3]) &
                (df[REORDERING_COL] == key[4])
            )
            df.loc[mask, base_col] = base_val
            df.loc[mask, speed_col] = base_val / df.loc[mask, metric]

    return df


In [ ]:
gcn_df = pd.concat([
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gcn_no_reordering", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gcn_torch_native", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gcn_cusoarse_with_precompute_bwd", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gcn_partition", index_col=0),
])
gcn_df[HEADS_COL] = 1

# min_aggr_new_fp32
# MIN AGGR:
min_aggr_df = pd.concat([
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/min_aggr_no_reordering", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/min_aggr_partition", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/min_aggr_new_fp32", index_col=0),
])
min_aggr_df[HEADS_COL] = 1

#gatv2

gatv2_df = pd.concat([
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/test_graph_reordering", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gat_v2_no_reordering", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gatv2_partition", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gat_v2_cuda_new_1", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gat_v2_cuda_new_2", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gat_v2_cuda_new_3", index_col=0),
])


# graph transformer
gt_df = pd.concat([
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gt_new", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gt_cuda_new", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gt_partition", index_col=0),
    pd.read_csv("/home/fvelikon/projects/cuda_exp/out/gt_dfgnn", index_col=0),
])


df = pd.concat([
    gcn_df,
    min_aggr_df,
    gatv2_df,
    gt_df,
])

In [27]:
df[MS_TOTAL] = df[FWD_MS_COL] + df[BWD_MS_COL]
df = add_speedup_columns(df, BASELINE)
df["density"] = df["num_edges"] / df["num_nodes"] / (df["num_nodes"] - 1)
df["average_node_degree"] = df["num_edges"] / df["num_nodes"]


In [28]:
def build_speedup_table_with_dataset_index(
    df_speed: pd.DataFrame,
    baseline: str,
) -> pd.DataFrame:
    """
    Build a single speedup table with a MultiIndex including dataset.

    Index:
        ('conv_type', 'dataset', 'heads', 'feature_dim')  if 'heads' in df_speed.columns
        ('conv_type', 'dataset', 'feature_dim')          otherwise

    Columns:
        MultiIndex (metric_short, backend), where metric_short ∈
        {'Fwd time', 'Bwd time', 'Fwd mem', 'Bwd mem'} and values are
        speedup vs baseline (>1 = better than baseline).

    Baseline backend column is dropped (speedup is always 1.0).
    """
    metrics = [
        (FWD_MS_COL,  "Fwd time"),
        (BWD_MS_COL,  "Bwd time"),
        (FWD_MEM_COL, "Fwd mem"),
        (BWD_MEM_COL, "Bwd mem"),
        (MS_TOTAL, "Fwd + Bwd time")
    ]

    # decide index cols
    idx_cols = [CONV_COL, DATASET_COL, HEADS_COL, FEATURE_DIM_COL]

    metric_frames = []

    for metric, short_name in metrics:
        speed_col = f"{metric}_speedup_vs_{baseline}"
        if speed_col not in df_speed.columns:
            continue

        df_metric = df_speed[np.isfinite(df_speed[speed_col])].copy()
        if df_metric.empty:
            continue

        pivot = df_metric.pivot_table(
            index=idx_cols,
            columns=BACKEND_COL,
            values=speed_col,
            aggfunc="max",
        )

        # drop baseline backend (always 1.0)
        pivot = pivot.drop(columns=[baseline], errors="ignore")

        if pivot.empty:
            continue

        # attach metric name as top-level column
        pivot.columns = pd.MultiIndex.from_product(
            [[short_name], pivot.columns],
            names=["metric", "backend"],
        )

        metric_frames.append(pivot)

    if not metric_frames:
        return pd.DataFrame()

    table = pd.concat(metric_frames, axis=1)

    # sort rows & columns
    table = table.sort_index()
    table = table.sort_index(axis=1)

    return table
def build_raw_table_with_dataset_index(
    df: pd.DataFrame,
    index = None,
) -> pd.DataFrame:
    """
    Build a raw-metrics table with the same structure as
    `build_speedup_table_with_dataset_index`, but containing
    *raw* runtimes (ms) and memory (MB) for all backends.

    Index:
        (conv_type, dataset, heads, feature_dim)

    Columns:
        MultiIndex (metric_short, backend), where metric_short ∈
        {'Fwd time', 'Bwd time', 'Fwd mem', 'Bwd mem', 'Fwd + Bwd time'}.

    Values:
        Raw numbers from the corresponding metric columns.
    """
    metrics = [
        (FWD_MS_COL,  "Fwd time"),
        (BWD_MS_COL,  "Bwd time"),
        (FWD_MEM_COL, "Fwd mem"),
        (BWD_MEM_COL, "Bwd mem"),
        (MS_TOTAL,    "Fwd + Bwd time"),
    ]

    idx_cols = index or [CONV_COL, DATASET_COL, HEADS_COL, FEATURE_DIM_COL]

    metric_frames: list[pd.DataFrame] = []

    for metric, short_name in metrics:
        if metric not in df.columns:
            continue

        df_metric = df[np.isfinite(df[metric])].copy()
        if df_metric.empty:
            continue

        pivot = df_metric.pivot_table(
            index=idx_cols,
            columns=BACKEND_COL,
            values=metric,
            aggfunc="min",  # same choice as in speedup table
        )

        if pivot.empty:
            continue

        # put metric name on top level
        pivot.columns = pd.MultiIndex.from_product(
            [[short_name], pivot.columns],
            names=["metric", "backend"],
        )

        metric_frames.append(pivot)

    if not metric_frames:
        return pd.DataFrame()

    table = pd.concat(metric_frames, axis=1)

    table = table.sort_index()
    table = table.sort_index(axis=1)

    return table


In [29]:
no_reordering_df = df[df[REORDERING_COL] == -1]
[
    [
        CONV_COL,
        FWD_MS_COL,
        BWD_MS_COL,
        FWD_MEM_COL,
        BWD_MEM_COL,
        DATASET_COL,
        BACKEND_COL,
        HEADS_COL,
        FEATURE_DIM_COL,
        REORDERING_COL,
    ]
]
reordering_df = df[df[REORDERING_COL] != -1]

[
    [
        CONV_COL,
        FWD_MS_COL,
        BWD_MS_COL,
        FWD_MEM_COL,
        BWD_MEM_COL,
        DATASET_COL,
        BACKEND_COL,
        HEADS_COL,
        FEATURE_DIM_COL,
        REORDERING_COL,
    ]
]

[['conv_type',
  'forward_ms',
  'backward_ms',
  'forward_memory_mb',
  'backward_memory_mb',
  'dataset',
  'backend',
  'heads',
  'feature_dim',
  'graph_reordering_partition_size']]

In [30]:
datasets_df = df.groupby(DATASET_COL)[["num_nodes", "num_edges"]].max().reset_index()
datasets_df["density"] = datasets_df["num_edges"] / datasets_df["num_nodes"] / (datasets_df["num_nodes"] - 1)
datasets_df["average_node_degree"] = datasets_df["num_edges"] / datasets_df["num_nodes"]

datasets_df = datasets_df.sort_values(by="density", ascending=False).reset_index(drop=True)

In [31]:
datasets_df

,dataset,num_nodes,num_edges,density,average_node_degree
0,hm-prices,46563,21461990,0.009899,460.923695
1,hm-categories,46563,21461990,0.009899,460.923695
2,tolokers-2,11758,1038000,0.007509,88.280320
3,avazu-ctr,76269,21968154,0.003777,288.035165
4,cora,2708,10556,0.001440,3.898080
5,citeseer,3327,9104,0.000823,2.736399
6,twitch-views,168114,13595114,0.000481,80.868423
7,pubmed,19717,88648,0.000228,4.496019
8,artnet-views,50405,560696,0.000221,11.123817
9,artnet-exp,50405,560696,0.000221,11.123817


In [32]:
datasets_df["dataset"].values.tolist()

['hm-prices',
 'hm-categories',
 'tolokers-2',
 'avazu-ctr',
 'cora',
 'citeseer',
 'twitch-views',
 'pubmed',
 'artnet-views',
 'artnet-exp',
 'city-reviews',
 'city-roads-M',
 'ogbn-arxiv',
 'ogbn-products',
 'city-roads-L',
 'pokec-regions',
 'web-fraud',
 'web-topics',
 'web-traffic']

In [33]:
px.box(df[(df[CONV_COL] == "gt") & (df[BACKEND_COL] != "dgl")], x=BACKEND_COL, y="ms_total_speedup_vs_dgl")

In [13]:
px.box(df[(df[CONV_COL] == "gt") & (df[BACKEND_COL] != "dgl")], x=BACKEND_COL, y="forward_ms_speedup_vs_dgl")

In [14]:
px.box(df[(df[CONV_COL] == "gcn") & (df[BACKEND_COL] != "dgl")], x=BACKEND_COL, y="forward_ms_speedup_vs_dgl")

In [15]:
px.box(df[(df[CONV_COL] == "gat_v2") & (df[BACKEND_COL] != "dgl")], x=BACKEND_COL, y="forward_ms_speedup_vs_dgl")

In [16]:
px.box(df[(df[CONV_COL] == "min_aggr") & (df[BACKEND_COL] != "dgl")], x=BACKEND_COL, y="forward_ms_speedup_vs_dgl")

## GT

In [17]:
no_reordering_df

,conv_type,dataset,backend,num_nodes,num_edges,avg_node_degree,feature_dim,forward_ms,forward_memory_mb,backward_ms,...,backward_ms_baseline_dgl,backward_ms_speedup_vs_dgl,forward_memory_mb_baseline_dgl,forward_memory_mb_speedup_vs_dgl,backward_memory_mb_baseline_dgl,backward_memory_mb_speedup_vs_dgl,ms_total_baseline_dgl,ms_total_speedup_vs_dgl,density,average_node_degree
0,gcn,artnet-exp,cusparse,50405,560696,0.089897,64,0.169984,44.418945,0.234086,...,0.413389,1.765967,91.189453,2.052941,118.342285,1.411166,1.647104,4.076280,0.000221,11.123817
1,gcn,artnet-exp,cusparse,50405,560696,0.089897,128,0.192717,71.554199,0.347750,...,0.469504,1.350118,140.412109,1.962318,192.176758,1.321805,1.635840,3.026715,0.000221,11.123817
2,gcn,artnet-exp,cusparse,50405,560696,0.089897,256,0.302285,122.330566,0.577331,...,0.677990,1.174353,241.964844,1.977959,344.090332,1.263502,1.849446,2.102561,0.000221,11.123817
3,gcn,artnet-exp,cusparse,50405,560696,0.089897,512,0.532480,219.225098,1.040486,...,1.121894,1.078240,435.753906,1.987701,634.773926,1.233608,2.592768,1.648330,0.000221,11.123817
4,gcn,artnet-exp,dgl,50405,560696,0.089897,64,1.233715,91.189453,0.413389,...,0.413389,1.000000,91.189453,1.000000,118.342285,1.000000,1.647104,1.000000,0.000221,11.123817
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,gt,pubmed,dfgnn,19717,88648,0.222419,256,1.082778,178.558594,2.828800,...,2.446131,0.864724,197.062012,1.103627,403.211426,1.194903,3.764429,0.962381,0.000228,4.496019
296,gt,pubmed,dfgnn,19717,88648,0.222419,256,1.173299,179.911133,2.823373,...,2.462106,0.872044,199.767090,1.110365,404.563965,1.194125,3.762586,0.941430,0.000228,4.496019
315,gt,pubmed,dfgnn,19717,88648,0.222419,512,2.279219,291.193848,4.914176,...,5.800448,1.180350,332.985840,1.143519,639.333008,1.056328,8.531456,1.186012,0.000228,4.496019
316,gt,pubmed,dfgnn,19717,88648,0.222419,512,2.288845,292.007324,4.909466,...,5.573018,1.135158,334.817383,1.146606,640.351074,1.056590,8.358912,1.161232,0.000228,4.496019


In [18]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gt"], "dgl").to_markdown(floatfmt=".2f"))

|                                 |   ('Bwd mem', 'cuda') |   ('Bwd mem', 'dfgnn') |   ('Bwd mem', 'triton_block_sparse') |   ('Bwd time', 'cuda') |   ('Bwd time', 'dfgnn') |   ('Bwd time', 'triton_block_sparse') |   ('Fwd + Bwd time', 'cuda') |   ('Fwd + Bwd time', 'dfgnn') |   ('Fwd + Bwd time', 'triton_block_sparse') |   ('Fwd mem', 'cuda') |   ('Fwd mem', 'dfgnn') |   ('Fwd mem', 'triton_block_sparse') |   ('Fwd time', 'cuda') |   ('Fwd time', 'dfgnn') |   ('Fwd time', 'triton_block_sparse') |
|:--------------------------------|----------------------:|-----------------------:|-------------------------------------:|-----------------------:|------------------------:|--------------------------------------:|-----------------------------:|------------------------------:|--------------------------------------------:|----------------------:|-----------------------:|-------------------------------------:|-----------------------:|------------------------:|-----------------------------------

In [19]:
print(build_raw_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gt"]).to_markdown(floatfmt=".2f"))

|                                 |   ('Bwd mem', 'cuda') |   ('Bwd mem', 'dfgnn') |   ('Bwd mem', 'dgl') |   ('Bwd mem', 'triton_block_sparse') |   ('Bwd time', 'cuda') |   ('Bwd time', 'dfgnn') |   ('Bwd time', 'dgl') |   ('Bwd time', 'triton_block_sparse') |   ('Fwd + Bwd time', 'cuda') |   ('Fwd + Bwd time', 'dfgnn') |   ('Fwd + Bwd time', 'dgl') |   ('Fwd + Bwd time', 'triton_block_sparse') |   ('Fwd mem', 'cuda') |   ('Fwd mem', 'dfgnn') |   ('Fwd mem', 'dgl') |   ('Fwd mem', 'triton_block_sparse') |   ('Fwd time', 'cuda') |   ('Fwd time', 'dfgnn') |   ('Fwd time', 'dgl') |   ('Fwd time', 'triton_block_sparse') |
|:--------------------------------|----------------------:|-----------------------:|---------------------:|-------------------------------------:|-----------------------:|------------------------:|----------------------:|--------------------------------------:|-----------------------------:|------------------------------:|----------------------------:|-------------------

In [20]:
print(build_raw_table_with_dataset_index(df[df[CONV_COL] == "gt"], [CONV_COL, DATASET_COL, HEADS_COL, FEATURE_DIM_COL, REORDERING_COL]).to_markdown(floatfmt=".2f"))

|                                       |   ('Bwd mem', 'cuda') |   ('Bwd mem', 'dfgnn') |   ('Bwd mem', 'dgl') |   ('Bwd mem', 'triton_block_sparse') |   ('Bwd time', 'cuda') |   ('Bwd time', 'dfgnn') |   ('Bwd time', 'dgl') |   ('Bwd time', 'triton_block_sparse') |   ('Fwd + Bwd time', 'cuda') |   ('Fwd + Bwd time', 'dfgnn') |   ('Fwd + Bwd time', 'dgl') |   ('Fwd + Bwd time', 'triton_block_sparse') |   ('Fwd mem', 'cuda') |   ('Fwd mem', 'dfgnn') |   ('Fwd mem', 'dgl') |   ('Fwd mem', 'triton_block_sparse') |   ('Fwd time', 'cuda') |   ('Fwd time', 'dfgnn') |   ('Fwd time', 'dgl') |   ('Fwd time', 'triton_block_sparse') |
|:--------------------------------------|----------------------:|-----------------------:|---------------------:|-------------------------------------:|-----------------------:|------------------------:|----------------------:|--------------------------------------:|-----------------------------:|------------------------------:|----------------------------:|-------

## GATv2

In [16]:
print(build_speedup_table_with_dataset_index(no_reordering_df[(no_reordering_df[CONV_COL] == "gat_v2") & (no_reordering_df[BACKEND_COL] != "pyg")], "dgl").to_markdown(floatfmt=".2f"))

|                                    |   ('Bwd mem', 'cuda') |   ('Bwd mem', 'cugraph') |   ('Bwd time', 'cuda') |   ('Bwd time', 'cugraph') |   ('Fwd + Bwd time', 'cuda') |   ('Fwd + Bwd time', 'cugraph') |   ('Fwd mem', 'cuda') |   ('Fwd mem', 'cugraph') |   ('Fwd time', 'cuda') |   ('Fwd time', 'cugraph') |
|:-----------------------------------|----------------------:|-------------------------:|-----------------------:|--------------------------:|-----------------------------:|--------------------------------:|----------------------:|-------------------------:|-----------------------:|--------------------------:|
| ('gat_v2', 'artnet-exp', 2, 256)   |                  5.49 |                     4.83 |                   1.60 |                      1.43 |                         1.96 |                            1.53 |                  8.42 |                     4.51 |                   3.23 |                      1.72 |
| ('gat_v2', 'artnet-exp', 2, 512)   |                  5.47 |  

In [17]:
print(build_raw_table_with_dataset_index(no_reordering_df[(no_reordering_df[CONV_COL] == "gat_v2") & (no_reordering_df[BACKEND_COL] != "pyg")]).to_markdown(floatfmt=".2f"))

|                                     |   ('Bwd mem', 'cuda') |   ('Bwd mem', 'cugraph') |   ('Bwd mem', 'dgl') |   ('Bwd time', 'cuda') |   ('Bwd time', 'cugraph') |   ('Bwd time', 'dgl') |   ('Fwd + Bwd time', 'cuda') |   ('Fwd + Bwd time', 'cugraph') |   ('Fwd + Bwd time', 'dgl') |   ('Fwd mem', 'cuda') |   ('Fwd mem', 'cugraph') |   ('Fwd mem', 'dgl') |   ('Fwd time', 'cuda') |   ('Fwd time', 'cugraph') |   ('Fwd time', 'dgl') |
|:------------------------------------|----------------------:|-------------------------:|---------------------:|-----------------------:|--------------------------:|----------------------:|-----------------------------:|--------------------------------:|----------------------------:|----------------------:|-------------------------:|---------------------:|-----------------------:|--------------------------:|----------------------:|
| ('gat_v2', 'artnet-exp', 2, 256)    |                925.47 |                  1052.09 |              5083.21 |             

## GCN

In [18]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gcn"], "dgl").to_markdown(floatfmt=".2f"))

|                                  |   ('Bwd mem', 'cusparse') |   ('Bwd mem', 'cusparse_precomputed_bwd') |   ('Bwd mem', 'fusegnn') |   ('Bwd mem', 'pyg') |   ('Bwd mem', 'tcgnn') |   ('Bwd mem', 'torch_native_gcn') |   ('Bwd mem', 'triton_block_sparse') |   ('Bwd time', 'cusparse') |   ('Bwd time', 'cusparse_precomputed_bwd') |   ('Bwd time', 'fusegnn') |   ('Bwd time', 'pyg') |   ('Bwd time', 'tcgnn') |   ('Bwd time', 'torch_native_gcn') |   ('Bwd time', 'triton_block_sparse') |   ('Fwd + Bwd time', 'cusparse') |   ('Fwd + Bwd time', 'cusparse_precomputed_bwd') |   ('Fwd + Bwd time', 'fusegnn') |   ('Fwd + Bwd time', 'pyg') |   ('Fwd + Bwd time', 'tcgnn') |   ('Fwd + Bwd time', 'torch_native_gcn') |   ('Fwd + Bwd time', 'triton_block_sparse') |   ('Fwd mem', 'cusparse') |   ('Fwd mem', 'cusparse_precomputed_bwd') |   ('Fwd mem', 'fusegnn') |   ('Fwd mem', 'pyg') |   ('Fwd mem', 'tcgnn') |   ('Fwd mem', 'torch_native_gcn') |   ('Fwd mem', 'triton_block_sparse') |   ('Fwd time', 'cus

In [19]:
print(build_raw_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "gcn"]).to_markdown(floatfmt=".2f"))

|                                  |   ('Bwd mem', 'cusparse') |   ('Bwd mem', 'cusparse_precomputed_bwd') |   ('Bwd mem', 'dgl') |   ('Bwd mem', 'fusegnn') |   ('Bwd mem', 'pyg') |   ('Bwd mem', 'tcgnn') |   ('Bwd mem', 'torch_native_gcn') |   ('Bwd mem', 'triton_block_sparse') |   ('Bwd time', 'cusparse') |   ('Bwd time', 'cusparse_precomputed_bwd') |   ('Bwd time', 'dgl') |   ('Bwd time', 'fusegnn') |   ('Bwd time', 'pyg') |   ('Bwd time', 'tcgnn') |   ('Bwd time', 'torch_native_gcn') |   ('Bwd time', 'triton_block_sparse') |   ('Fwd + Bwd time', 'cusparse') |   ('Fwd + Bwd time', 'cusparse_precomputed_bwd') |   ('Fwd + Bwd time', 'dgl') |   ('Fwd + Bwd time', 'fusegnn') |   ('Fwd + Bwd time', 'pyg') |   ('Fwd + Bwd time', 'tcgnn') |   ('Fwd + Bwd time', 'torch_native_gcn') |   ('Fwd + Bwd time', 'triton_block_sparse') |   ('Fwd mem', 'cusparse') |   ('Fwd mem', 'cusparse_precomputed_bwd') |   ('Fwd mem', 'dgl') |   ('Fwd mem', 'fusegnn') |   ('Fwd mem', 'pyg') |   ('Fwd mem', 'tcgn

In [38]:
print(build_raw_table_with_dataset_index(df[df[CONV_COL] == "gcn"], [CONV_COL, DATASET_COL, HEADS_COL, FEATURE_DIM_COL, REORDERING_COL]).to_markdown(floatfmt=".2f"))

|                                        |   ('Bwd mem', 'cusparse') |   ('Bwd mem', 'cusparse_precomputed_bwd') |   ('Bwd mem', 'dgl') |   ('Bwd mem', 'fusegnn') |   ('Bwd mem', 'pyg') |   ('Bwd mem', 'tcgnn') |   ('Bwd mem', 'torch_native_gcn') |   ('Bwd mem', 'triton_block_sparse') |   ('Bwd time', 'cusparse') |   ('Bwd time', 'cusparse_precomputed_bwd') |   ('Bwd time', 'dgl') |   ('Bwd time', 'fusegnn') |   ('Bwd time', 'pyg') |   ('Bwd time', 'tcgnn') |   ('Bwd time', 'torch_native_gcn') |   ('Bwd time', 'triton_block_sparse') |   ('Fwd + Bwd time', 'cusparse') |   ('Fwd + Bwd time', 'cusparse_precomputed_bwd') |   ('Fwd + Bwd time', 'dgl') |   ('Fwd + Bwd time', 'fusegnn') |   ('Fwd + Bwd time', 'pyg') |   ('Fwd + Bwd time', 'tcgnn') |   ('Fwd + Bwd time', 'torch_native_gcn') |   ('Fwd + Bwd time', 'triton_block_sparse') |   ('Fwd mem', 'cusparse') |   ('Fwd mem', 'cusparse_precomputed_bwd') |   ('Fwd mem', 'dgl') |   ('Fwd mem', 'fusegnn') |   ('Fwd mem', 'pyg') |   ('Fwd mem',

## MinAggr

In [21]:
print(build_speedup_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "min_aggr"], "dgl").to_markdown(floatfmt=".2f"))

|                                       |   ('Bwd mem', 'cuda') |   ('Bwd mem', 'cugraph') |   ('Bwd time', 'cuda') |   ('Bwd time', 'cugraph') |   ('Fwd + Bwd time', 'cuda') |   ('Fwd + Bwd time', 'cugraph') |   ('Fwd mem', 'cuda') |   ('Fwd mem', 'cugraph') |   ('Fwd time', 'cuda') |   ('Fwd time', 'cugraph') |
|:--------------------------------------|----------------------:|-------------------------:|-----------------------:|--------------------------:|-----------------------------:|--------------------------------:|----------------------:|-------------------------:|-----------------------:|--------------------------:|
| ('min_aggr', 'artnet-exp', 1, 64)     |                  1.41 |                     1.11 |                   1.26 |                      0.79 |                         1.67 |                            0.89 |                  1.67 |                     1.16 |                   2.26 |                      0.96 |
| ('min_aggr', 'artnet-exp', 1, 128)    |              

In [21]:
print(build_raw_table_with_dataset_index(no_reordering_df[no_reordering_df[CONV_COL] == "min_aggr"]).to_markdown(floatfmt=".2f"))

|                                       |   ('Bwd mem', 'cuda') |   ('Bwd mem', 'cugraph') |   ('Bwd mem', 'dgl') |   ('Bwd time', 'cuda') |   ('Bwd time', 'cugraph') |   ('Bwd time', 'dgl') |   ('Fwd + Bwd time', 'cuda') |   ('Fwd + Bwd time', 'cugraph') |   ('Fwd + Bwd time', 'dgl') |   ('Fwd mem', 'cuda') |   ('Fwd mem', 'cugraph') |   ('Fwd mem', 'dgl') |   ('Fwd time', 'cuda') |   ('Fwd time', 'cugraph') |   ('Fwd time', 'dgl') |
|:--------------------------------------|----------------------:|-------------------------:|---------------------:|-----------------------:|--------------------------:|----------------------:|-----------------------------:|--------------------------------:|----------------------------:|----------------------:|-------------------------:|---------------------:|-----------------------:|--------------------------:|----------------------:|
| ('min_aggr', 'artnet-exp', 1, 64)     |                 94.94 |                   120.73 |               133.97 |       

In [22]:
import pandas as pd


In [23]:
gt_kernel_ncu = pd.read_csv("../gt_profile.csv")
gt_kernel_ncu

,ID,Process ID,Process Name,Host Name,Kernel Name,Context,Stream,Block Size,Grid Size,Device,...,smsp__warps_active.min.peak_sustained,smsp__warps_active.min.per_cycle_active,smsp__warps_active.sum.peak_sustained,smsp__warps_active.sum.per_cycle_active,smsp__warps_eligible.avg.per_cycle_active,smsp__warps_eligible.max.per_cycle_active,smsp__warps_eligible.min.per_cycle_active,smsp__warps_eligible.sum.per_cycle_active,thread_inst_executed,thread_inst_executed_true
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,warp,warp,warp,warp,warp,warp,warp,warp,inst,inst
1,0.0,2235973.0,python3.11,127.0.0.1,"GraphAttentionForward_CSR_MH(unsigned long, un...",1.0,7.0,"(32, 1, 1)","(500, 1, 1)",0.0,...,16,0.66,6912,495.97,0.09,0.24,0.04,40.20,26531000,25394500
2,1.0,2235973.0,python3.11,127.0.0.1,"GraphAttentionForward_CSR_MH(unsigned long, un...",1.0,7.0,"(32, 1, 1)","(500, 1, 1)",0.0,...,16,0.65,6912,496.06,0.09,0.24,0.04,40.10,26531000,25394500


In [13]:
df_test = pd.read_csv("../test_partition", index_col=0)

In [18]:
df_test

,conv_type,dataset,backend,num_nodes,num_edges,avg_node_degree,feature_dim,heads,forward_ms,forward_memory_mb,backward_ms,backward_memory_mb,graph_reordering_partition_size,device_name,device_total_memory_mb,sm_count,compute_capability,max_threads_per_sm,registers_per_sm,experiment_name
0,gt,artnet-views,dgl,50405,560696,0.089897,512,2,6.940263,746.346191,14.908826,1443.795898,-1,NVIDIA A100-SXM4-80GB,81152.75,108,8.0,2048,65536,gt_dgl_artnet-views_feature_dim_512_heads_2
1,gt,artnet-views,dgl,50405,560696,0.089897,512,2,6.948045,744.975586,14.963609,1442.425293,32,NVIDIA A100-SXM4-80GB,81152.75,108,8.0,2048,65536,gt_dgl_artnet-views_feature_dim_512_heads_2
2,gt,artnet-views,dgl,50405,560696,0.089897,512,2,6.960128,744.975586,14.965555,1442.425293,64,NVIDIA A100-SXM4-80GB,81152.75,108,8.0,2048,65536,gt_dgl_artnet-views_feature_dim_512_heads_2
3,gt,artnet-views,dgl,50405,560696,0.089897,512,2,6.961766,744.975586,14.965248,1442.425293,128,NVIDIA A100-SXM4-80GB,81152.75,108,8.0,2048,65536,gt_dgl_artnet-views_feature_dim_512_heads_2
4,gt,artnet-views,dgl,50405,560696,0.089897,512,2,6.948864,744.975586,15.013991,1442.425293,256,NVIDIA A100-SXM4-80GB,81152.75,108,8.0,2048,65536,gt_dgl_artnet-views_feature_dim_512_heads_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,gt,twitch-views,triton_block_sparse,168114,13595114,0.012366,512,2,34.644992,2901.062500,399.700586,4546.763672,512,NVIDIA A100-SXM4-80GB,81152.75,108,8.0,2048,65536,gt_triton_block_sparse_twitch-views_feature_di...
176,gt,twitch-views,triton_block_sparse,168114,13595114,0.012366,512,2,32.287949,2896.116699,429.360010,4541.817871,1024,NVIDIA A100-SXM4-80GB,81152.75,108,8.0,2048,65536,gt_triton_block_sparse_twitch-views_feature_di...
177,gt,twitch-views,triton_block_sparse,168114,13595114,0.012366,512,2,34.328168,2890.816406,422.754297,4536.517578,4096,NVIDIA A100-SXM4-80GB,81152.75,108,8.0,2048,65536,gt_triton_block_sparse_twitch-views_feature_di...
178,gt,twitch-views,triton_block_sparse,168114,13595114,0.012366,512,2,33.651303,2881.969238,385.971704,4527.670410,8192,NVIDIA A100-SXM4-80GB,81152.75,108,8.0,2048,65536,gt_triton_block_sparse_twitch-views_feature_di...


In [21]:
df_test["graph_reordering_partition_size"] = df_test["graph_reordering_partition_size"].astype(str)

In [22]:
for dataset in df_test[DATASET_COL].unique():
    df_dataset = df_test[df_test[DATASET_COL] == dataset]
    fig = px.scatter(df_dataset, x="graph_reordering_partition_size", y=FWD_MS_COL, title=dataset, color=BACKEND_COL)
    fig.show()